# Phase 12 — Ecoli Feature Engineering Expansion

**Project:** “Machine Learning approaches for accelerating antibiotic susceptibility testing (AST) in microfluidic systems” 

**Date:** May 2026 

## 1. Introduction

This notebook presents the additional feature engineering process applied to GVR measurements, deriving time-normalised metrics, cross-tube statistical summaries, and growth-related indicators to capture both intra-sample variability and overall experimental dynamics.

### Objectives
- Generate additional features from the original GVR measurements
- Capture temporal growth dynamics through time-normalised metrics
- Extract experiment-level and tube-level summary statistics
- Provide a richer representation of bacterial growth behaviour

### New Features
| Feature | Description |
|---|---|
| `auc_per_time` | AUC normalised by time |
| `delta_per_time` | Delta normalised by time |
| `derivative_per_time` | Derivative normalised by time |
| `mean_gvr` | Mean GVR across all tubes at timepoint t |
| `max_gvr_tube` | Cumulative max GVR of the tube up to t |
| `min_gvr_tube` | Cumulative min GVR of the tube up to t |
| `max_gvr_experiment` | Max GVR across all tubes at timepoint t |
| `min_gvr_experiment` | Min GVR across all tubes at timepoint t |
| `growth_score` | Percentage change in GVR from t-1 to t |
| `growth_score_experiment` | Percentage change in mean GVR across tubes |

## Table of Contents
1. [Introduction](#1-introduction)
2. [Imports & Setup](#2-imports--setup)
3. [Data Loading](#3-data-loading)
4. [Dataset Parsing](#4-dataset-parsing)
5. [Feature Engineering](#5-feature-engineering)
6. [Label Assignment](#6-label-assignment)
7. [Quality Check](#7-quality-check)
8. [Save Dataset](#8-save-dataset)

## 2. Imports & Setup

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Serialisation
import pickle

# Project utilities
from Preprocessing_functions_1 import *

## 3. Data Loading

### 3.1 Load feature matrix

In [2]:
# Load the base feature matrix from Phase 2 (without label)
for_ml = pd.read_csv("ecoli_data_ml.csv")
for_ml = for_ml.drop(columns=["label"])

print(f"Dataset shape: {for_ml.shape}")
for_ml.head()

Dataset shape: (45342, 13)


,experiencia,tubo_id,antibiotico,concentração,tempo,gvr,std_inicial,std_atual,std_tubo,slope,delta,AUC,Derivada
0,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.000000,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.000000,0.000000
1,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.166667,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.011647,0.000000
2,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.333333,-0.139765,0.008784,0.017568,0.098829,-0.628944,-0.209648,0.005824,-1.257887
3,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.500000,-0.199132,0.008784,0.025127,0.121492,-0.538028,-0.269014,-0.022418,-0.356198
4,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.666667,-0.119479,0.008784,0.024350,0.112185,-0.284042,-0.189362,-0.048969,0.477916


### 3.2 Load raw dataset

In [3]:
with open("ecoli_data.pkl", "rb") as f:
    ecoli_data = pickle.load(f)

print("Raw dataset loaded successfully.")

Raw dataset loaded successfully.


## 4. Dataset Parsing

In [4]:
# Extract experiments, tube IDs, time series and GVR values
registos = []

for antibiotico in ecoli_data:
    for date in ecoli_data[antibiotico]:
        for concentration in ecoli_data[antibiotico][date]:
            df = ecoli_data[antibiotico][date][concentration]["gvr"]
            chave = antibiotico + "_" + date + "_" + concentration
            antibiotic = antibiotico
            
            # Melt wide → long (one row per timepoint per antibiotic concentration)
            colunas = [c for c in df.columns if c != "Time(hrs)"]
            
            df_temp = df.melt(
                id_vars="Time(hrs)",
                value_vars=colunas,
                var_name="concentração",
                value_name="gvr"
            ).rename(columns={"Time(hrs)": "tempo"})
            
            df_temp["experiencia"] = chave
            df_temp["tubo_id"]     = chave + "_" + df_temp["concentração"].astype(str)
            df_temp["antibiotico"] = antibiotic
            
            registos.append(df_temp)

for_ml = pd.concat(registos, ignore_index=True)[
    ["experiencia", "tubo_id", "antibiotico", "concentração", "tempo", "gvr"]
]

display(for_ml)

,experiencia,tubo_id,antibiotico,concentração,tempo,gvr
0,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.000000,0.069883
1,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.166667,0.069883
2,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.333333,-0.139765
3,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.500000,-0.199132
4,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.666667,-0.119479
...,...,...,...,...,...,...
45575,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.333333,0.648233
45576,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.500000,0.663801
45577,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.666667,0.657226
45578,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.833333,0.688446


In [5]:
for antiibiotico in ecoli_data:
    for data in ecoli_data[antiibiotico]:
        for concetração_bacteriana in ecoli_data[antiibiotico][data]:
            print(f"antibiotico={antiibiotico} | date={data} | concentration={concetração_bacteriana}")

antibiotico=ampicillin | date=20230508 | concentration=5e6
antibiotico=ampicillin | date=20230508 | concentration=5e5
antibiotico=ampicillin | date=20230508 | concentration=5e4
antibiotico=ampicillin | date=20230516 | concentration=5e6
antibiotico=ampicillin | date=20230516 | concentration=5e5
antibiotico=ampicillin | date=20230516 | concentration=5e4
antibiotico=ampicillin | date=20230518 | concentration=5e6
antibiotico=ampicillin | date=20230518 | concentration=5e5
antibiotico=ampicillin | date=20230518 | concentration=5e4
antibiotico=ciprofloxacin | date=20250717 | concentration=5e5
antibiotico=ciprofloxacin | date=20250719 | concentration=5e5
antibiotico=ciprofloxacin | date=20250728 | concentration=5e5
antibiotico=gentamicin | date=20250717 | concentration=5e5
antibiotico=gentamicin | date=20250719 | concentration=5e5
antibiotico=gentamicin | date=20250728 | concentration=5e5
antibiotico=tetracycline | date=20240118 | concentration=5e6
antibiotico=tetracycline | date=20240118 | co

In [6]:
for_ml = for_ml.drop_duplicates(subset=["experiencia", "concentração", "tempo"]).reset_index(drop=True)

## 5. Feature Engineering

New features are computed at each timepoint, capturing both tube-level and experiment-level growth dynamics.

In [7]:
# Compute the initial cross-tube std for each experiment (t=0)
stds = {}

for exp, grupo in for_ml.groupby("experiencia"):

    anti, date, concentration = exp.split("_")
    
    # Precisas do antibiotico — está no grupo
    antibiotico = grupo["antibiotico"].iloc[0]
    
    df = ecoli_data[antibiotico][date][concentration]["gvr"]
    
    std_inicial = np.std(df.iloc[0, 1:])

    stds[exp] = std_inicial

for_ml["std_inicial"] = for_ml["experiencia"].map(stds)

### 5.1 Growth score helper function

In [8]:
def get_slope(gvr_inicial, gvr_atual, tempo):
    """Rate of change from the initial GVR value."""
    if tempo == 0:
        return 0
    resultado = (gvr_atual - gvr_inicial)/tempo
    return resultado

def get_delta(gvr_inicial, gvr_atual):
    """Absolute change from the initial GVR value."""
    return gvr_atual - gvr_inicial 

def get_auc(gvr_ateagora, tempo_ateagora):
    """Area under the GVR curve up to the current timepoint."""
    auc_parcial = np.trapz(gvr_ateagora, tempo_ateagora)
    return auc_parcial

def get_derivada(gvr_serie, tempo_serie, t):
    #Instantaneous derivative at timepoint t.
    if t == 0:
        return 0
    return (gvr_serie[t] - gvr_serie[t-1]) / (tempo_serie[t] - tempo_serie[t-1])

def get_growth_score(anterior, atual):
    """Percentage change in GVR from the previous to the current timepoint."""
    return ((atual - anterior) / anterior) * 100

In [9]:
stds_por_tempo = {}

for exp, grupo in for_ml.groupby("experiencia"):
    anti, date, concentration = exp.split("_")
    antibiotico = grupo["antibiotico"].iloc[0]
    
    df = ecoli_data[antibiotico][date][concentration]["gvr"]
    
    tempos = sorted(grupo["tempo"].unique())
    for t in tempos:
        row = df[df["Time(hrs)"].round(6) == round(t, 6)]
        if len(row) > 0:
            stds_por_tempo[(exp, round(t, 6))] = np.std(df.loc[row.index[0], df.columns[1:]])

for (experiencia, concentração), grupo in for_ml.groupby(["experiencia", "concentração"]):
    idx = grupo.sort_values("tempo").index
    gvr_serie   = grupo.sort_values("tempo")["gvr"].values
    tempo_serie = grupo.sort_values("tempo")["tempo"].values
    gvr_inicial = gvr_serie[0]

    stds_exp = {k[1]: v for k, v in stds_por_tempo.items() if k[0] == experiencia}
    tempos_std = sorted(stds_exp.keys())

    for t, i in enumerate(idx):
        t_key = round(tempo_serie[t], 6)
        std_val = stds_exp.get(t_key, stds_exp[tempos_std[-1]])

        for_ml.at[i, "std_atual"] = std_val
        for_ml.at[i, "std_tubo"]  = np.std(gvr_serie[:t+1])
        for_ml.at[i, "slope"]     = get_slope(gvr_inicial, gvr_serie[t], tempo_serie[t])
        for_ml.at[i, "delta"]     = get_delta(gvr_inicial, gvr_serie[t])
        for_ml.at[i, "AUC"]       = get_auc(gvr_serie[:t+1], tempo_serie[:t+1])
        for_ml.at[i, "Derivada"]  = get_derivada(gvr_serie, tempo_serie, t)
        #Novas Features
        for_ml.at[i, "gvr_max_tubo"]  = (max(gvr_serie[:t+1])) 
        for_ml.at[i, "gvr_min_tubo"]     = (min(gvr_serie[:t+1])) 

        t_key = round(tempo_serie[t], 6)

        row = df[df["Time(hrs)"].round(6) == t_key]

        if len(row) == 0:
            for_ml.at[i, "gvr_media"] = np.nan
            for_ml.at[i, "gvr_max_experiencia"] = np.nan
            for_ml.at[i, "gvr_min_experiencia"] = np.nan
            for_ml.at[i, "growth_score"] = np.nan
            for_ml.at[i, "growth_score_exp"] = np.nan
        else:
            for_ml.at[i, "gvr_max_experiencia"] = row.iloc[0, 1:].max()
            for_ml.at[i, "gvr_min_experiencia"] = row.iloc[0, 1:].min()
            for_ml.at[i, "growth_score"] = row.iloc[0, 1:].max()
            for_ml.at[i, "growth_score_exp"] = row.iloc[0, 1:].min()
            for_ml.at[i, "gvr_media"] = row.iloc[0, 1:].mean()


In [10]:
for_ml['auc_por_tempo'] = for_ml['AUC'] / for_ml['tempo']
for_ml['delta_por_tempo'] = for_ml['delta'] / for_ml['tempo']
for_ml['Derivada_por_tempo'] = for_ml['Derivada'] / for_ml['tempo']

nan_cols = ["auc_por_tempo", "delta_por_tempo", "Derivada_por_tempo", "growth_score", "growth_score_exp", "gvr_max_experiencia", "gvr_min_experiencia", "gvr_media"]
for_ml[nan_cols] = for_ml[nan_cols].fillna(0)

In [11]:
display(for_ml)

,experiencia,tubo_id,antibiotico,concentração,tempo,gvr,std_inicial,std_atual,std_tubo,slope,...,gvr_max_tubo,gvr_min_tubo,gvr_max_experiencia,gvr_min_experiencia,growth_score,growth_score_exp,gvr_media,auc_por_tempo,delta_por_tempo,Derivada_por_tempo
0,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.000000,0.069883,0.008784,0.008784,0.000000,0.000000,...,0.069883,0.069883,0.055765,-0.032625,0.055765,-0.032625,0.020375,0.000000,0.000000,0.000000
1,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.166667,0.069883,0.008784,0.008784,0.000000,0.000000,...,0.069883,0.069883,0.047773,-0.036017,0.047773,-0.036017,0.000335,0.069883,0.000000,0.000000
2,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.333333,-0.139765,0.008784,0.017568,0.098829,-0.628944,...,0.069883,-0.139765,-0.005548,-0.042579,-0.005548,-0.042579,-0.020710,0.017471,-0.628944,-3.773662
3,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.500000,-0.199132,0.008784,0.025127,0.121492,-0.538028,...,0.069883,-0.199132,0.016940,-0.061092,0.016940,-0.061092,-0.022240,-0.044836,-0.538028,-0.712396
4,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0,0.666667,-0.119479,0.008784,0.024350,0.112185,-0.284042,...,0.069883,-0.199132,0.010814,-0.035092,0.010814,-0.035092,-0.007005,-0.073453,-0.284042,0.716874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45337,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.333333,0.648233,0.023087,1.163355,0.271984,0.031497,...,0.724460,-0.018297,3.021870,-0.188806,3.021870,-0.188806,0.819495,0.389183,0.031497,-0.005971
45338,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.500000,0.663801,0.023087,1.156492,0.272000,0.032000,...,0.724460,-0.018297,2.989244,-0.179602,2.989244,-0.179602,0.824378,0.391352,0.032000,0.004556
45339,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.666667,0.657226,0.023087,1.162899,0.271947,0.031424,...,0.724460,-0.018297,3.002420,-0.190713,3.002420,-0.190713,0.834016,0.393523,0.031424,-0.001909
45340,trimethoprim_20250728_5e5,trimethoprim_20250728_5e5_0.06,trimethoprim,0.06,20.833333,0.688446,0.023087,1.206671,0.272131,0.032671,...,0.724460,-0.018297,3.192166,-0.192478,3.192166,-0.192478,0.840795,0.395758,0.032671,0.008991


### 5.3 Add normalised and new features to DataFrame

## 6. Label Assignment

In [12]:
# Assign binary growth label based on final AUC per tube
for id_tubo in for_ml["tubo_id"].unique():
    subset = for_ml[for_ml["tubo_id"] == id_tubo]
    ultimo = subset.iloc[-1]
    
    label = "1" if ultimo["AUC"] > 0 else "0"
    
    for_ml.loc[for_ml["tubo_id"] == id_tubo, "label"] = label


## 7. Quality Check

In [13]:
# Check for missing values
print("Missing values per column:")
print(for_ml.isnull().sum())

# Inspect first row
print(f"\nDataset shape: {for_ml.shape}")
display(for_ml.iloc[0])

Missing values per column:
experiencia            0
tubo_id                0
antibiotico            0
concentração           0
tempo                  0
gvr                    0
std_inicial            0
std_atual              0
std_tubo               0
slope                  0
delta                  0
AUC                    0
Derivada               0
gvr_max_tubo           0
gvr_min_tubo           0
gvr_max_experiencia    0
gvr_min_experiencia    0
growth_score           0
growth_score_exp       0
gvr_media              0
auc_por_tempo          0
delta_por_tempo        0
Derivada_por_tempo     0
label                  0
dtype: int64

Dataset shape: (45342, 24)


experiencia              ampicillin_20230508_5e6
tubo_id                ampicillin_20230508_5e6_0
antibiotico                           ampicillin
concentração                                   0
tempo                                        0.0
gvr                                     0.069883
std_inicial                             0.008784
std_atual                               0.008784
std_tubo                                     0.0
slope                                        0.0
delta                                        0.0
AUC                                          0.0
Derivada                                     0.0
gvr_max_tubo                            0.069883
gvr_min_tubo                            0.069883
gvr_max_experiencia                     0.055765
gvr_min_experiencia                    -0.032625
growth_score                            0.055765
growth_score_exp                       -0.032625
gvr_media                               0.020375
auc_por_tempo       

## 8. Save Dataset

In [14]:
# Save expanded feature matrix for use in Phase 6
for_ml.to_csv("ecoli_data_expanded.csv", index=False)

print("Dataset saved successfully.")
print(f"  Shape  : {for_ml.shape}")
print(f"  Labels : {for_ml['label'].value_counts().to_dict()}")

Dataset saved successfully.
  Shape  : (45342, 24)
  Labels : {'1': 27768, '0': 17574}
